# 2. Evaluate — original CRM data (`data/crm/`)

Same steps as `python -m src.evaluate --output reports/crm/final`, using the functions in `src/evaluate.py`.
Loads the models saved by `1_train.ipynb`; nothing here can change the selected model, threshold or calibration.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from src import evaluate, train
from src.data.crm import CRM_DIR


## Settings

Must match the settings used in `1_train.ipynb`.

In [ ]:
DATA_DIR = CRM_DIR   # real Maven CRM tables
QUICK = False            # True: tiny budgets, smoke test only (writes a *smoke* folder)
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

## 1. Load the saved models and rebuild the same split

In [ ]:
ev = evaluate.start(OUTPUT)
print("Data:", ev.out.source_data())
print("Selected model:", ev.trained.selected, "| mode:", ev.manifest["mode"])
print("Test deals:", len(ev.split.test))

## 2. Test metrics

Each model uses its own validation threshold. The Dummy control shows what predicting the majority class gives.

In [ ]:
COLUMNS = ["model", "roc_auc", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "brier", "tn", "fp", "fn", "tp"]
scores = evaluate.score_test(ev)[COLUMNS].round(4)

# Every (feature set, model) pair on the same test rows, after the validation lock.
comparison = evaluate.compare_feature_sets(ev)
display(comparison[["feature_set", "model", "selected", "validation_roc_auc", "test_roc_auc", "test_brier"]].round(4))
scores

### Calibration of the selected model

Calibration changes the probability scale (Brier), not the ranking (ROC-AUC).

In [ ]:
pd.read_csv(ev.out.metrics / "calibration_test.csv")[["variant", "roc_auc", "brier", "log_loss", "accuracy", "f1", "threshold"]].round(4)

## 3. Figures

In [ ]:
evaluate.make_figures(ev)
display(Image(filename=str(ev.out.figures / "roc_curves.png"), width=600))
display(Image(filename=str(ev.out.figures / "reliability.png"), width=500))
display(Image(filename=str(ev.out.figures / f"confusion_{ev.trained.selected}.png"), width=350))

## 4. Explanations

Permutation importance: ROC-AUC drop on test when one input is shuffled. TreeSHAP: raw Random Forest only.
Both describe the model's behaviour, not causes.

In [ ]:
evaluate.explain_models(ev)
display(pd.read_csv(ev.out.explain / "permutation_importance_test.csv").head(10).round(4))
pd.read_csv(ev.out.explain / "rf_shap_global.csv").head(10).round(4)

## 5. Audits

The close-value model is deliberately invalid: it shows how an outcome field fakes a near-perfect score.

In [ ]:
evaluate.run_audits(ev)
display(pd.read_csv(ev.out.checks / "leakage_audit.csv")[["model", "roc_auc", "accuracy", "purpose"]].round(4))
pd.read_csv(ev.out.checks / "priority_test.csv").round(4)

## 6. Novelty checks

Compares the as-of split with a no-purge and a random split, and checks that the explanations describe the model.

In [ ]:
protocols, checks = evaluate.run_checks(ev)
display(protocols[["protocol", "model", "roc_auc_mean", "roc_auc_sd", "delta_vs_asof"]].round(4))
pd.json_normalize(checks, sep=".").T

## 7. Record the stage

In [ ]:
manifest = evaluate.finish(ev)

Next: `3_predict.ipynb`.